# Traffic Matrix Prediction — Results Analysis

Run `python src/train.py --data data/traffic_matrix.csv --window 12 --epochs 50` first — this notebook reads the artifacts from `results/`.

In [ ]:
import json, os, sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd() / 'src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RESULTS = Path('results')
with open(RESULTS / 'metrics.json') as f:
    metrics = json.load(f)
preds = np.load(RESULTS / 'predictions.npz')
Y_true = preds['Y_true']
print('Models:', list(metrics.keys()))
print('Y_true shape:', Y_true.shape)

## 1. MSE comparison (replicates paper Fig. 8)

In [ ]:
order = [n for n in ['ARMA','LinearRegression','RandomForest','FFNN','LSTM'] if n in metrics]
mse_vals = [metrics[n]['MSE'] for n in order]

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(order, mse_vals, color=['#c44','#e9a','#6c6','#69c','#34a'])
ax.set_ylabel('MSE (scaled)'); ax.set_title('Prediction methods — MSE comparison')
ax.set_yscale('log')
for i, v in enumerate(mse_vals):
    ax.text(i, v, f'{v:.4f}', ha='center', va='bottom')
plt.grid(axis='y', alpha=0.3, which='both')
plt.tight_layout(); plt.show()

## 2. Predicted vs actual traffic for a high-volume OD pair

In [ ]:
mean_flow = Y_true.mean(axis=0)
high_idx = int(np.argmax(mean_flow))
n_nodes = int(round(np.sqrt(Y_true.shape[1])))
i, j = divmod(high_idx, n_nodes)

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(Y_true[:, high_idx], label='actual', color='black', linewidth=2)
for name in order:
    if name in preds.files:
        ax.plot(preds[name][:, high_idx], label=name, alpha=0.75)
ax.set_title(f'Highest-volume OD: node {i} → node {j}')
ax.set_xlabel('test time step'); ax.set_ylabel('traffic volume (scaled)')
ax.grid(alpha=0.3); ax.legend()
plt.tight_layout(); plt.show()

## 3. LSTM per-OD error heatmap

Which origin-destination pairs is the LSTM worst at?

In [ ]:
if 'LSTM' in preds.files:
    per_od = np.mean((Y_true - preds['LSTM'])**2, axis=0).reshape(n_nodes, n_nodes)
    plt.figure(figsize=(6.5, 5.5))
    plt.imshow(per_od, cmap='magma')
    plt.colorbar(label='MSE')
    plt.xlabel('destination node j'); plt.ylabel('origin node i')
    plt.title('LSTM per-OD MSE')
    plt.tight_layout(); plt.show()
    worst = np.unravel_index(np.argmax(per_od), per_od.shape)
    print(f'worst OD: {worst}  (MSE = {per_od[worst]:.4f})')

## 4. LSTM training curves

In [ ]:
with open(RESULTS / 'lstm_history.json') as f:
    hist = json.load(f)
plt.figure(figsize=(8, 4))
plt.plot(hist['loss'], label='train')
if 'val_loss' in hist: plt.plot(hist['val_loss'], label='val')
plt.xlabel('epoch'); plt.ylabel('MSE'); plt.title('LSTM training curves')
plt.grid(alpha=0.3); plt.legend()
plt.tight_layout(); plt.show()